In [ ]:
import pandas as pd
import uuid
from pathlib import Path
import json

In [ ]:
RAW_DATA_DIR = Path("../data/01_raw")

In [ ]:
df_processed = pd.read_csv(RAW_DATA_DIR/"labeled_dataset_example.csv")
df_processed.head()

In [ ]:
def generate_sentence_id():
    """Generate a unique sentence ID similar to the example format"""
    return f"cmhtk{uuid.uuid4().hex[:20]}"

def parse_leaf_code_to_annotations(leaf_code):
    """
    Convert a leaf code like '2512.1' into hierarchical annotations.
    Example: '2512.1' -> [
        {"level": 1, "nodeCode": "2"},
        {"level": 2, "nodeCode": "25"},
        {"level": 3, "nodeCode": "251"},
        {"level": 4, "nodeCode": "2512"},
        {"level": 5, "nodeCode": "2512.1"}
    ]
    """
    # Remove decimal point and handle the code as a string
    code_str = str(leaf_code).replace('.', '')
    annotations = []
    
    # Build hierarchical codes progressively
    # Level 1: first digit
    if len(code_str) >= 1:
        annotations.append({"level": 1, "nodeCode": code_str[0]})
    
    # Level 2: first two digits
    if len(code_str) >= 2:
        annotations.append({"level": 2, "nodeCode": code_str[:2]})
    
    # Level 3: first three digits
    if len(code_str) >= 3:
        annotations.append({"level": 3, "nodeCode": code_str[:3]})
    
    # Level 4: first four digits
    if len(code_str) >= 4:
        annotations.append({"level": 4, "nodeCode": code_str[:4]})
    
    # Level 5: full code with decimal (if exists)
    if '.' in str(leaf_code) and len(code_str) > 4:
        annotations.append({"level": 5, "nodeCode": str(leaf_code)})
    
    return annotations

# Test the function with an example
test_code = "2512.1"
print(f"Testing with code: {test_code}")
print(parse_leaf_code_to_annotations(test_code))

In [ ]:
# Convert dataframe to the required JSON structure
sentences = []

for _, row in df_processed.iterrows():
    sentence = {
        "sentenceId": generate_sentence_id(),
        "fields": {
            "job_description": row['text']
        },
        "annotations": parse_leaf_code_to_annotations(row['leaf_code'])
    }
    sentences.append(sentence)

# Create the final dictionary
sentences_dict = {
    "sentences": sentences
}

# Display first sentence as example
print("First sentence example:")
print(json.dumps(sentences_dict["sentences"][0], indent=2))
print(f"\nTotal sentences: {len(sentences_dict['sentences'])}")

In [ ]:
with open(RAW_DATA_DIR/"isco_training_sentences.json", "w") as f:
    json.dump(sentences_dict, f, indent=2)